# Preprocessing glacier datasets

In [ ]:
import os 
import glob
import numpy as np
import pandas as pd
import geopandas as gpd

# Import libraries
import rioxarray as rio
from rasterio.enums import Resampling

os.chdir('/home/rooda/Datasets/GLACIERS') 

#TODO CHECK FINAL RES AND RESAMPLING METHOD

## 1. **Hugonnet et al. 2021**
- Merges multiple glacier elevation change rate (dhdt) raster files
- Reprojects data to EPSG:32718 (UTM Zone 18S) and then to EPSG:4326 (WGS84)
- Resamples to 100m resolution
- Converts ice thickness to water equivalent (mm) using density factor (1.091)
- **Output**: `dhdt_2000_2020_hugonnet.tif` (glacier mass balance 2000-2020)


In [ ]:
## dhdt
list_files = glob.glob("DHDT/*.tif")

glacier_list = []

# Read rasters file
for glacier in list_files:
    glacier_i = rio.open_rasterio(glacier, chunks = "auto")
    glacier_i = glacier_i.rio.reproject("EPSG:32718")
    glacier_list.append(glacier_i)

# Merge/Mosaic multiple rasters using merge_arrays method of rioxarray
merged_raster = rio.merge.merge_arrays(dataarrays = glacier_list, res = (100, 100), crs="EPSG:32718", method='max')
merged_raster = merged_raster.where(merged_raster != -9999, np.nan)
merged_raster = merged_raster.rio.write_nodata(np.nan)

merged_raster = merged_raster.rio.reproject("EPSG:4326")
merged_raster = merged_raster * 1000 / 1.091  # from ice to water equivalent (mm)
merged_raster = merged_raster.fillna(0)
merged_raster.rio.to_raster("dhdt_2000_2020_hugonnet.tif", compress="LZW")


### 2. **Millan et al. 2022**
- Processes glacier volume data from multiple tiles
- Reprojects to EPSG:32718 with bilinear resampling, then to EPSG:4326
- Merges tiles at 100m resolution
- Converts units from meters to km³
- **Output**: `Volume_Millan_2022_100m.tif`


In [ ]:
list_files = glob.glob("VOLUME/Millan_2022/*.tif")
glacier_list = []

# Read rasters file
for glacier in list_files:
    glacier_i = rio.open_rasterio(glacier)
    glacier_i = glacier_i.rio.reproject("EPSG:32718", resampling = Resampling.bilinear)
    glacier_list.append(glacier_i)

# Merge/Mosaic multiple rasters using merge_arrays method of rioxarray
merged_raster = rio.merge.merge_arrays(dataarrays = glacier_list, res = (100, 100), crs="EPSG:32718", method='max')
merged_raster = merged_raster.where(merged_raster != -9999, np.nan)
merged_raster = merged_raster.rio.write_nodata(np.nan)
merged_raster = merged_raster * 100**2 / 1e9 # from m to km3
merged_raster = merged_raster.rio.reproject("EPSG:4326")
merged_raster = merged_raster.fillna(0)
merged_raster.rio.to_raster("Volume_Millan_2022_100m.tif", compress="LZW")


### 3. **Farinotti et al. 2019**
- Processes ice thickness data for RGI regions 16 (Low Latitudes) and 17 (Southern Andes)
- Filters glaciers to study area (longitude: -80° to -50°, latitude: < 0°)
- Reprojects to EPSG:32718 at 200m resolution
- Processes each RGI region separately
- **Output**: `Thickness_Farinotti_2019_[region].tif`

In [ ]:
list_files = glob.glob("VOLUME/Farinotti_2019/*.tif")

# subset glaciers
RGI6_16 = gpd.read_file("RGI/RGI60_16.shp")
RGI6_16 = RGI6_16[(RGI6_16['CenLon'] >= -80) & (RGI6_16['CenLon'] <= -50)] # remove glaciers outside the study area
RGI6_16 = RGI6_16[RGI6_16['CenLat'] < 0] 
RGI6_17 = gpd.read_file("RGI/RGI60_17.shp")

for RGI in [RGI6_16, RGI6_17]:
    list_files_df = pd.DataFrame(list_files, columns=['dir'])
    list_files_df['RGIId'] = list_files_df['dir'].str.extract(r'(RGI60-\d{2}\.\d{5})')
    list_files_df = list_files_df[list_files_df['RGIId'].isin(RGI.RGIId)]

    glacier_list = []
    for glacier in list_files_df.dir:
        glacier_i = rio.open_rasterio(glacier)
        glacier_i = glacier_i.rio.reproject("EPSG:32718", resolution = (200, 200), resampling = Resampling.bilinear)
        glacier_list.append(glacier_i)

    merged_raster = rio.merge.merge_arrays(dataarrays = glacier_list, res = (200, 200), crs="EPSG:32718", method='max')
    merged_raster = merged_raster.where(merged_raster != -9999, np.nan)
    merged_raster = merged_raster.rio.write_nodata(np.nan)
    merged_raster.rio.to_raster("Thickness_Farinnoti_2019_{}.tif".format(RGI.O1Region[0]), compress="LZW")  